# DeepWeeds Lab Day 2 - 2A202603012 Bui Quang Vinh

Bật GPU, đặt `images.zip` đúng thư mục Drive, rồi chạy từ SECTION 00.
Notebook tự khôi phục output và checkpoint khi phiên bị ngắt.

Chỉ chọn cấu hình bằng validation. Test chạy sau **FINAL CONFIG LOCK**.
Trạng thái hiện tại: **CODE READY; EXPERIMENTS PENDING**.

## SECTION 00 - Kiểm tra GPU

Cell này cần runtime GPU. Thông tin phần cứng được lưu cùng kết quả.

In [ ]:
import gc
import importlib
import json
import os
import platform
import shutil
import subprocess
import sys
from dataclasses import replace
from pathlib import Path

import torch

if not torch.cuda.is_available():
    raise RuntimeError("Chọn runtime GPU trong Colab rồi chạy lại từ SECTION 00.")

DEVICE = "cuda"
GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", GPU_NAME)
print("VRAM GiB:", round(GPU_VRAM_GB, 2))

## SECTION 01 - Kết nối Drive

File dữ liệu cần nằm ở `MyDrive/K4_Track4_Day2/data/images.zip`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/K4_Track4_Day2")
SOURCE_DIR = DRIVE_ROOT / "source"

for folder in (
    "data",
    "source",
    "runs",
    "checkpoints",
    "curves",
    "predictions",
    "eval_out",
    "results",
    "submission",
):
    (DRIVE_ROOT / folder).mkdir(parents=True, exist_ok=True)

## SECTION 02 - Nạp code và thư viện

Code chạy từ local Colab. Chạy lại cell này khi nguồn trên Drive được cập nhật.

In [ ]:
WORK_DIR = Path("/content/deepweeds")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

CODE_DIR = WORK_DIR / "code"
CODE_DIR.mkdir(exist_ok=True)
REQUIRED_FILES = (
    "dataset.py",
    "model.py",
    "losses.py",
    "train.py",
    "inference.py",
    "benchmark.py",
    "eval.py",
    "artifacts.py",
    "workflow.py",
    "sanity.py",
    "reporting.py",
    "runtime_checks.py",
    "requirements-colab.txt",
    "README.md",
)
missing = [name for name in REQUIRED_FILES if not (SOURCE_DIR / name).is_file()]
if missing:
    raise FileNotFoundError("Thiếu file trong Drive source/: " + ", ".join(missing))

for name in REQUIRED_FILES:
    shutil.copy2(SOURCE_DIR / name, CODE_DIR / name)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(CODE_DIR / "requirements-colab.txt"),
    ],
    check=True,
)

In [ ]:
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
importlib.invalidate_caches()

import matplotlib.pyplot as plt
import pandas as pd
import timm
from PIL import Image

import eval as ev
import dataset
import model
import losses
import inference
import benchmark
import train
import artifacts
import reporting
import sanity
import workflow

for module in (
    ev,
    dataset,
    model,
    losses,
    inference,
    benchmark,
    train,
    artifacts,
    reporting,
    sanity,
    workflow,
):
    importlib.reload(module)
    if Path(module.__file__).resolve().parent != CODE_DIR.resolve():
        raise RuntimeError("Module được nạp từ sai thư mục: " + module.__name__)

workflow.restore_outputs(DRIVE_ROOT)
print("Code đã nạp:", CODE_DIR)
print("timm:", timm.__version__)
print("pandas:", pd.__version__)

## SECTION 03 - CSV fold 0 chính thức

Tải và kiểm tra checksum: `labels.csv`, `train_subset0.csv`, `val_subset0.csv`, `test_subset0.csv`.
Nguồn có một ảnh lệch nhãn giữa train và `labels.csv`; giữ nguyên nhãn split và ghi lại ngoại lệ.

In [ ]:
DATA_DIR = WORK_DIR / "data"
LABELS_DIR = DATA_DIR / "labels"
IMAGE_DIR = DATA_DIR / "images"
ZIP_PATH = DRIVE_ROOT / "data" / "images.zip"

train_df, val_df, test_df = workflow.prepare_labels(LABELS_DIR)
all_filenames = pd.concat([train_df, val_df, test_df], ignore_index=True).Filename
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

## SECTION 04 - Chuẩn bị ảnh local

Kiểm tra ZIP và chỉ giải nén ảnh còn thiếu. Ảnh không được đọc trực tiếp từ Drive lúc train.

In [ ]:
workflow.prepare_images(ZIP_PATH, IMAGE_DIR, all_filenames)
split_report = dataset.check_split(train_df, val_df, test_df, IMAGE_DIR)
workflow.save_json(DRIVE_ROOT / "results" / "fold0_split_check.json", split_report)
print(json.dumps(split_report, ensure_ascii=False, indent=2))

## SECTION 05 - Xem dữ liệu

Lưu phân bố lớp và ba ảnh mẫu mỗi lớp để dùng trong báo cáo.

In [ ]:
counts = pd.DataFrame(split_report["per_class"]).T
counts.plot(kind="bar", figsize=(14, 5), title="DeepWeeds fold 0")
plt.ylabel("Number of images")
plt.tight_layout()
plt.savefig("curves/fold0_class_counts.png", dpi=150)
plt.show()
shutil.copy2(
    "curves/fold0_class_counts.png", DRIVE_ROOT / "curves" / "fold0_class_counts.png"
)

image_stats = {}
for filename in train_df.Filename.head(100):
    with Image.open(IMAGE_DIR / filename) as image:
        key = str((image.size, image.mode))
    image_stats[key] = image_stats.get(key, 0) + 1
print("Kích thước và chế độ màu của 100 ảnh train:", image_stats)

In [ ]:
fig, axes = plt.subplots(9, 3, figsize=(10, 24))
for class_id, class_name in enumerate(dataset.CLASS_NAMES):
    rows = train_df[train_df.Label == class_id].head(3)
    for column, row in enumerate(rows.itertuples()):
        with Image.open(IMAGE_DIR / row.Filename) as image:
            axes[class_id, column].imshow(image.convert("RGB"))
        axes[class_id, column].set_title(class_name)
        axes[class_id, column].axis("off")

fig.tight_layout()
fig.savefig("curves/fold0_examples.png", dpi=130)
plt.show()
shutil.copy2("curves/fold0_examples.png", DRIVE_ROOT / "curves" / "fold0_examples.png")

## SECTION 06 - Kiểm tra code trước khi train

Bộ kiểm tra dùng dữ liệu nhỏ, không ghi số liệu thí nghiệm vào kết quả bài lab.
Nếu cell này lỗi, dừng để sửa trước khi chạy screening.

In [ ]:
subprocess.run(
    [sys.executable, "-X", "utf8", str(CODE_DIR / "runtime_checks.py")],
    check=True,
)

In [ ]:
probe_loader = dataset.make_loader(
    train_df.head(8),
    IMAGE_DIR,
    dataset.build_transforms(True, 224),
    batch_size=8,
    train=True,
    num_workers=0,
    seed=0,
)
images, labels, filenames = next(iter(probe_loader))
sanity_result = sanity.run(images, labels, device=DEVICE)
workflow.save_json("sanity_checks.json", sanity_result)
workflow.save_json(DRIVE_ROOT / "results" / "sanity_checks.json", sanity_result)
print(json.dumps(sanity_result, indent=2))

mean = torch.tensor(dataset.IMAGENET_MEAN).view(1, 3, 1, 1)
std = torch.tensor(dataset.IMAGENET_STD).view(1, 3, 1, 1)
augmented = (images * std + mean).clamp(0, 1)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for axis, image, label in zip(axes.flat, augmented, labels):
    axis.imshow(image.permute(1, 2, 0).numpy())
    axis.set_title(dataset.CLASS_NAMES[int(label)])
    axis.axis("off")
fig.tight_layout()
fig.savefig("curves/augmentation_examples.png", dpi=140)
plt.show()
shutil.copy2(
    "curves/augmentation_examples.png",
    DRIVE_ROOT / "curves" / "augmentation_examples.png",
)
del probe_loader, images, labels, augmented
gc.collect()
torch.cuda.empty_cache()

## SECTION 07 - Cấu hình nền

Dùng batch 32 và 10 epoch cho mọi backbone. `FORCE_RERUN=False` giữ cơ chế skip/resume.
Không đổi cấu hình sau khi đã có kết quả test.

In [ ]:
FORCE_RERUN = False
BASE = train.Config(
    seed=0,
    epochs=10,
    batch_size=32,
    img_size=224,
    amp=True,
    inference_amp=False,
    num_workers=2,
    images_dir=str(IMAGE_DIR),
    labels_dir=str(LABELS_DIR),
    out_dir="runs",
    pred_dir="predictions",
    curve_dir="curves",
    sync_dir=str(DRIVE_ROOT),
)
print(BASE)

## SECTION 08 - Screening năm backbone

Cùng recipe, seed và epoch. Sau mỗi epoch, checkpoint và log được sync về Drive.

In [ ]:
backbone_specs = [
    ("B01", "resnet50"),
    ("B02", "resnext50_32x4d"),
    ("B03", "convnext_tiny"),
    ("B04", "deit_small_patch16_224"),
    ("B05", "efficientnet_b0"),
]
backbone_results = []
for tag, name in backbone_specs:
    cfg = replace(BASE, exp_id=tag, backbone=name)
    summary = workflow.run_or_resume(cfg, force=FORCE_RERUN)
    backbone_results.append(summary)

## SECTION 09 - So sánh backbone

Đo latency ban đầu và chọn backbone bằng validation macro-F1.

In [ ]:
for summary in backbone_results:
    if summary.get("latency_p95_ms") is None:
        cfg = workflow.read_run_config(summary)
        net = workflow.load_model_for_run(cfg, DEVICE)
        timing = benchmark.method_latency_report(net, "fp32", amp=False, device=DEVICE)
        summary["latency_p95_ms"] = timing["p95"]
        workflow.save_json(train.run_dir(cfg) / "summary.json", summary)
        workflow.save_json(DRIVE_ROOT / train.run_dir(cfg) / "summary.json", summary)
        del net
        torch.cuda.empty_cache()

backbone_winner = workflow.choose_training_winner(backbone_results)
WINNER_BACKBONE = workflow.read_run_config(backbone_winner).backbone
print("Backbone thắng trên validation:", WINNER_BACKBONE)
display(pd.DataFrame(backbone_results))

## SECTION 10 - Ablation đơn yếu tố

T00 dùng backbone winner. Mỗi cấu hình tiếp theo đổi một yếu tố: khởi tạo, augmentation, loss hoặc EMA.

In [ ]:
BASELINE = replace(BASE, exp_id="T00", backbone=WINNER_BACKBONE)
ablation_specs = [
    ("T00", {}),
    ("T01", {"init": "frozen"}),
    ("T02", {"init": "scratch"}),
    ("T03", {"aug": "color"}),
    ("T04", {"aug": "trivial"}),
    ("T05", {"loss": "ls", "label_smoothing": 0.1}),
    ("T06", {"loss": "focal", "focal_gamma": 2.0}),
    ("T07", {"mix": "cutmix", "mix_alpha": 1.0}),
    ("T08", {"ema_decay": 0.999}),
]
ablation_results = []
for tag, updates in ablation_specs:
    cfg = replace(BASELINE, exp_id=tag, **updates)
    summary = workflow.run_or_resume(cfg, force=FORCE_RERUN)
    ablation_results.append(summary)

## SECTION 11 - Kết hợp yếu tố thắng

Chọn augmentation, loss và EMA từ validation; chạy một cấu hình kết hợp.

In [ ]:
augmentation_rows = [
    row for row in ablation_results if row["exp_id"] in {"T00", "T03", "T04", "T07"}
]
loss_rows = [row for row in ablation_results if row["exp_id"] in {"T00", "T05", "T06"}]
ema_rows = [row for row in ablation_results if row["exp_id"] in {"T00", "T08"}]

best_aug = workflow.read_run_config(workflow.choose_training_winner(augmentation_rows))
best_loss = workflow.read_run_config(workflow.choose_training_winner(loss_rows))
best_ema = workflow.read_run_config(workflow.choose_training_winner(ema_rows))
combo = replace(
    BASELINE,
    exp_id="T_COMBO",
    aug=best_aug.aug,
    mix=best_aug.mix,
    mix_alpha=best_aug.mix_alpha,
    loss=best_loss.loss,
    label_smoothing=best_loss.label_smoothing,
    focal_gamma=best_loss.focal_gamma,
    ema_decay=best_ema.ema_decay,
)
ablation_results.append(workflow.run_or_resume(combo, force=FORCE_RERUN))
selected = workflow.choose_training_winner(ablation_results)
SELECTED_CFG = workflow.read_run_config(selected)
display(pd.DataFrame(ablation_results))
print("Recipe thắng trên validation:", SELECTED_CFG)

## SECTION 12 - Inference trên validation

So sánh FP32/AMP, TTA, cách gộp, độ phân giải và temperature scaling.
Tự lưu từng phương pháp để tiếp tục sau khi phiên bị ngắt.

In [ ]:
inference_rows = workflow.inference_experiments(SELECTED_CFG, val_df, DEVICE)
display(pd.DataFrame(inference_rows))

## SECTION 13 - FINAL CONFIG LOCK

Khóa recipe, inference và seed dựa trên validation. Từ đây mới cho phép phase test.

In [ ]:
lock = workflow.lock_final(SELECTED_CFG, BASELINE, inference_rows, DRIVE_ROOT)
print(json.dumps(lock, ensure_ascii=False, indent=2))

## SECTION 14 - Final và baseline với ba seed

Giữ train/val riêng. Mỗi seed chạy test một lần; file test đã có được kiểm tra và bỏ qua.

In [ ]:
for seed in lock["seeds"]:
    for is_final in (False, True):
        cfg = workflow.final_config(
            lock, seed, is_final, DRIVE_ROOT, IMAGE_DIR, LABELS_DIR
        )
        workflow.run_or_resume(cfg)

## SECTION 15 - Latency cấu hình cuối

Đo đúng recipe inference đã khóa trên checkpoint F01 seed 0, với batch 1 và 32.

In [ ]:
cfg = workflow.final_config(lock, 0, True, DRIVE_ROOT, IMAGE_DIR, LABELS_DIR)
latency = workflow.final_latency(cfg, DEVICE)
print(json.dumps(latency, indent=2))

## SECTION 16 - Chấm bằng eval.py

Giữ nguyên eval.py. Chấm final và baseline rồi tính mức cải thiện.

In [ ]:
EVAL_DIR = DRIVE_ROOT / "eval_out"
for tag in ("T00", "F01"):
    arguments = [
        "score",
        "--pred",
        f"predictions/{tag}_seed*_test.csv",
        "--test-csv",
        str(LABELS_DIR / "test_subset0.csv"),
        "--labels",
        str(LABELS_DIR / "labels.csv"),
        "--tag",
        tag,
        "--out",
        str(EVAL_DIR),
    ]
    if ev.main(arguments) != 0:
        raise RuntimeError("eval.py score thất bại: " + tag)

grade_args = [
    "grade",
    "--final",
    "predictions/F01_seed*_test.csv",
    "--baseline",
    "predictions/T00_seed*_test.csv",
    "--final-val",
    "predictions/F01_seed*_val.csv",
    "--val-csv",
    str(LABELS_DIR / "val_subset0.csv"),
    "--test-csv",
    str(LABELS_DIR / "test_subset0.csv"),
    "--labels",
    str(LABELS_DIR / "labels.csv"),
    "--out",
    str(EVAL_DIR),
    "--latency-p95-ms",
    str(latency[0]["p95"]),
    "--latency-method",
    "proper",
]
if all(
    Path(f"predictions/F01uncal_seed{seed}_test.csv").exists() for seed in lock["seeds"]
):
    grade_args.extend(["--uncal", "predictions/F01uncal_seed*_test.csv"])
if ev.main(grade_args) != 0:
    raise RuntimeError("eval.py grade thất bại; xem thông báo phía trên.")

## SECTION 17 - Phân tích lỗi

Báo precision/recall/F1 từng seed và mean/std; không thay bằng ensemble ngoài cấu hình đã khóa.

In [ ]:
per_class = reporting.analyze_tests(
    pred_dir="predictions",
    labels_dir=LABELS_DIR,
    out_dir=EVAL_DIR,
    images_dir=IMAGE_DIR,
)
workflow.save_json(DRIVE_ROOT / "results" / "per_class.json", per_class)
for name in (
    "T00_seed0_confusion.png",
    "F01_seed0_confusion.png",
    "cross_class_error_examples.png",
):
    source = EVAL_DIR / name
    if source.exists():
        shutil.copy2(source, DRIVE_ROOT / "curves" / name)

display(pd.DataFrame(per_class).query("seed == 'mean'"))

## SECTION 18 - Tạo Excel và báo cáo

Chỉ lấy số từ log và prediction đã lưu. Các trường chưa có giữ PENDING.

In [ ]:
workbook = DRIVE_ROOT / "results" / "results.xlsx"
artifacts.generate_results(workbook, latency_file="latency.json")
report_path = DRIVE_ROOT / "results" / "report.md"
artifacts.generate_report(report_path, workbook)
print("Excel:", workbook)
print("Báo cáo:", report_path)

## SECTION 19 - Đóng gói bài nộp

Gói chỉ gồm code, báo cáo, Excel, curves và prediction; không chứa dataset hay checkpoint lớn.

In [ ]:
submission = DRIVE_ROOT / "submission" / "2A202603012_BuiQuangVinh"
for folder in ("code", "curves", "predictions"):
    (submission / folder).mkdir(parents=True, exist_ok=True)

shutil.copy2(SOURCE_DIR / "README.md", submission / "README.md")
shutil.copy2(workbook, submission / "results.xlsx")
shutil.copy2(report_path, submission / "report.md")
for source in (DRIVE_ROOT / "curves").glob("*.png"):
    shutil.copy2(source, submission / "curves" / source.name)
for source in Path("predictions").glob("*.csv"):
    shutil.copy2(source, submission / "predictions" / source.name)
for source in SOURCE_DIR.iterdir():
    if source.is_file() and source.suffix in {".py", ".ipynb", ".txt"}:
        shutil.copy2(source, submission / "code" / source.name)

archive = shutil.make_archive(
    str(submission), "zip", submission.parent, submission.name
)
print("Thư mục bài nộp:", submission)
print("File ZIP:", archive)